#### Оператор «морж» (`:=`)

In [ ]:
# Без моржа — лишние вычисления или переменная
data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# ❌ Многословно
filtered = [x for x in data if x > 5]
if len(filtered) > 3:
    print(f"Найдено {len(filtered)} элементов")   # вычисляется дважды

# ✅ Морж — присваивание внутри выражения
if (n := len([x for x in data if x > 5])) > 3:
    print(f"Найдено {n} элементов")   # n уже связана

# Морж в цикле while — типичный паттерн для потокового чтения
import io
source = io.StringIO("line1\nline2\nline3")
while chunk := source.readline():
    print(chunk.strip())

#### Распаковка

In [ ]:
# Базовая распаковка
first, *rest = [10, 20, 30, 40, 50]
print(first)  # 10
print(rest)   # [20, 30, 40, 50]

# Обмен значениями без временной переменной
a, b = 1, 2
a, b = b, a

# Распаковка в for
records = [("Alice", 85), ("Bob", 92), ("Carol", 78)]
for name, score in records:
    print(f"{name}: {score}")

# Распаковка словаря / слияние (Python 3.9+)
defaults = {"timeout": 30, "retries": 3}
overrides = {"retries": 5, "verbose": True}
config = {**defaults, **overrides}
print(config)  # {'timeout': 30, 'retries': 5, 'verbose': True}

#### `*args` и `**kwargs`

In [ ]:
def summarize(*values, label="Итого", precision=2):
    total = sum(values)
    avg = total / len(values)
    return f"{label}: сумма={total:.{precision}f}, среднее={avg:.{precision}f}"

print(summarize(10, 20, 30, label="Выручка Q1"))
# Выручка Q1: сумма=60.00, среднее=20.00

# Проброс kwargs — полезно в обёртках
def log_and_call(func, **kwargs):
    print(f"Вызываем {func.__name__} с параметрами {kwargs}")
    return func(**kwargs)

#### Декораторы

In [ ]:
import time
import functools

# Простой декоратор-таймер
def timer(func):
    @functools.wraps(func)  # сохраняем имя и документацию оригинальной функции
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} выполнялась {elapsed:.4f}с")
        return result
    return wrapper

@timer
def load_data(path: str) -> list:
    time.sleep(0.1)  # имитация I/O
    return []

load_data("data.csv")  # load_data выполнялась 0.1003с

# Декоратор с аргументами
def retry(max_attempts=3):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"Попытка {attempt} не удалась: {e}")
            raise RuntimeError(f"Не удалось после {max_attempts} попыток")
        return wrapper
    return decorator

@retry(max_attempts=3)
def fetch_data(url: str):
    raise ConnectionError("Сеть недоступна")  # имитация сбоя

#### Базовые аннотации

In [ ]:
from typing import Optional, Union, Literal
from collections.abc import Sequence

# Простые аннотации (в Python 3.10+ вместо Union используется |)
def compute_tax(amount: float, rate: float = 0.20) -> float:
    return amount * rate

# Optional — значение или None
def find_customer(customer_id: int) -> Optional[dict]:
    db = {1: {"name": "Alice"}}
    return db.get(customer_id)   # возвращает dict или None

# Union (Python 3.10+: float | str)
def format_amount(value: Union[float, str]) -> str:
    return f"${float(value):,.2f}"

# Literal — ограниченный набор строковых значений
Status = Literal["pending", "approved", "rejected"]
def update_status(order_id: int, status: Status) -> None:
    print(f"Заказ {order_id} → {status}")

# Sequence — принимает list, tuple и т.д.
def average(values: Sequence[float]) -> float:
    return sum(values) / len(values)

#### `@dataclass` — структурированные данные без шаблонного кода

In [ ]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Transaction:
    id: int
    category: str
    amount: float
    currency: str = "USD"
    tags: list[str] = field(default_factory=list)   # безопасное изменяемое значение по умолчанию
    note: Optional[str] = None

    @property
    def amount_eur(self) -> float:
        rates = {"USD": 0.92, "GBP": 1.15}
        return self.amount * rates.get(self.currency, 1.0)

t = Transaction(id=1, category="electronics", amount=500.0)
print(t)                 # Transaction(id=1, category='electronics', ...)
print(t.amount_eur)      # 460.0
print(t == Transaction(id=1, category="electronics", amount=500.0))  # True (авто __eq__)

#### [demo] Подсчитать суммарные продажи по категориям из списка транзакций.

In [ ]:
transactions = [
    {"category": "electronics", "amount": 500},
    {"category": "food",        "amount": 30},
    {"category": "electronics", "amount": 200},
    {"category": "food",        "amount": 45},
    {"category": "clothing",    "amount": 120},
]

#### 1. Процедурный стиль

In [ ]:
# Явный, императивный, удобно отлаживать пошагово
def compute_totals_procedural(transactions):
    totals = {}
    for t in transactions:
        cat = t["category"]
        if cat not in totals:
            totals[cat] = 0.0
        totals[cat] += t["amount"]
    return totals

print(compute_totals_procedural(transactions))
# {'electronics': 700.0, 'food': 75.0, 'clothing': 120.0}

#### 2. Функциональный стиль

In [ ]:
from functools import reduce

def compute_totals_functional(transactions):
    def reducer(acc, t):
        cat = t["category"]
        return {**acc, cat: acc.get(cat, 0) + t["amount"]}

    return reduce(reducer, transactions, {})

# Или с генератором словаря + groupby
from itertools import groupby

def compute_totals_groupby(transactions):
    sorted_txns = sorted(transactions, key=lambda t: t["category"])
    return {
        cat: sum(t["amount"] for t in group)
        for cat, group in groupby(sorted_txns, key=lambda t: t["category"])
    }

print(compute_totals_functional(transactions))

#### 3. ООП стиль

In [ ]:
from collections import defaultdict

class SalesAggregator:
    def __init__(self):
        self._totals: dict[str, float] = defaultdict(float)
        self._counts: dict[str, int] = defaultdict(int)

    def process(self, transaction: dict) -> "SalesAggregator":
        cat = transaction["category"]
        self._totals[cat] += transaction["amount"]
        self._counts[cat] += 1
        return self   # текучий интерфейс для цепочек вызовов

    def summary(self) -> dict:
        return {
            cat: {"total": total, "count": self._counts[cat], "avg": total / self._counts[cat]}
            for cat, total in self._totals.items()
        }

agg = SalesAggregator()
for t in transactions:
    agg.process(t)
print(agg.summary())

#### Иерархия исключений

In [ ]:
# Собственные исключения — дают понятные сообщения об ошибках
class DataPipelineError(Exception):
    """Базовый класс для всех ошибок пайплайна."""

class MissingFieldError(DataPipelineError):
    def __init__(self, field: str, row_id):
        self.field = field
        self.row_id = row_id
        super().__init__(f"Строка {row_id}: отсутствует обязательное поле '{field}'")

class InvalidAmountError(DataPipelineError):
    pass

# Использование с try/except/else/finally
def parse_transaction(row: dict) -> dict:
    try:
        amount = float(row["amount"])
        if amount < 0:
            raise InvalidAmountError(f"Отрицательная сумма: {amount}")
    except KeyError:
        raise MissingFieldError("amount", row.get("id", "?"))
    except ValueError:
        raise InvalidAmountError(f"Нечисловая сумма: {row['amount']!r}")
    else:
        # выполняется только если исключений не было
        return {"category": row["category"], "amount": amount}
    finally:
        # выполняется всегда — используйте для очистки / логирования
        pass   # например, закрыть соединение с БД

#### Контекстные менеджеры

In [ ]:
# Контекстный менеджер на основе класса
class Timer:
    def __enter__(self):
        import time
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self.start
        print(f"Прошло: {self.elapsed:.4f}с")
        return False   # не подавлять исключения

with Timer() as t:
    result = sum(range(1_000_000))
print(f"Сумма: {result}")

# На основе генератора (проще)
from contextlib import contextmanager

@contextmanager
def managed_connection(path):
    conn = open(path)
    try:
        yield conn
    finally:
        conn.close()
        print("Соединение закрыто")

with managed_connection("data.csv") as f:
    print(f.readline())